# 8 — `fold`: the tensor-state scan

Notebook 07 ended with structured-state reducers: a fold whose accumulator
is a tuple of *scalars*, enough for h_t = a_t·h_{t−1} + b_t. The state that
real time-stepped computation carries is a **tensor** — an SSM's matrix
state, a PDE's field pair — and its step is not a scalar expression but a
whole computation.

`fold` is that combinator, and its step is an **IR Program**: programs
become first-class values inside programs. Denotationally it is
`List.foldl step init elems` with per-step emission — deliberately the most
proof-friendly control structure available, since every property about it
is one induction over the step list rather than a statement about an
n-instruction unrolled blob. Two things fall out of that choice:

- the **carry contract** — the step must return each state at *exactly* its
  incoming layout, checked rather than coerced (D17: the library never
  silently re-aligns);
- the **derived adjoint** — `grad` of a fold is a reverse fold over the
  step's VJP program, obtained by running `grad` on the step itself. The
  correctness argument is modular: if `grad` is right on the step, the
  reverse fold is right on the fold, by induction over steps.

Sequential by definition. An associative tensor *combine* — the license for
chunk-parallel evaluation — is a declaration a compiler may exploit later;
it does not change this denotation.

In [1]:
import nbhelp  # noqa: F401  — puts tensorlib on sys.path
import numpy as np
from pdum.tl import Tensor, defreducer, ops_count, u
from pdum.tl.autodiff import grad, numeric_grad
from pdum.tl.ir import Instr, Program, run
from pdum.tl.signatures import SignatureError, infer_signatures

In [2]:
def instr(var, op, operands=(), **params):
    return Instr(var, op, tuple(operands), params)


def T(arr, names):
    return Tensor.from_numpy(np.asarray(arr, dtype=np.float64), names)


def validate(prog, inputs, wrt, target="loss"):
    """Finite-difference check of one gradient, the way notebook 05 does it."""
    joint, grads = grad(prog, target, dict(inputs))
    env = run(joint, inputs)
    got = env[grads[wrt]].to_numpy(order=inputs[wrt].names)
    want = numeric_grad(prog, target, wrt, inputs)
    ok = np.allclose(got, want, rtol=2e-5, atol=1e-7)
    print(("OK " if ok else "FAIL") + f"  d{target}/d{wrt:<3} max|Δ| = {np.abs(got - want).max():.2e}")
    return joint, grads, env


rng = np.random.default_rng(23)

## Anatomy

A `fold` instruction takes k state inits followed by m element tensors, and
these params:

| param | meaning |
|---|---|
| `step` | an IR `Program`; its `input` vars are the state names ++ element names |
| `dim` | the scan dim (chartless in the reference — `strip_charts` first, glue charts back after) |
| `state` | step-input names receiving the carried state (order matches the first k operands) |
| `element` | step-input names receiving per-step SLICES of the element operands (the scan dim is select-ed away) |
| `carry` | {state name → step var producing the next state}; must preserve the state's layout |
| `out` | `("emit", v)` stacks step var v along the scan dim; `("final", v)` returns a carry's last value |
| `extent` | `(start, stop)`, required only when there are no elements |

Start where notebook 07 left off: the scalar recurrence, written as a step
program instead of a combine tree.

In [3]:
step = Program(
    (
        instr("h", "input"),
        instr("av", "input"),
        instr("bv", "input"),
        instr("ah", "pointwise", ["av", "h"], f="mul"),
        instr("h1", "pointwise", ["ah", "bv"], f="add"),
    )
)
print(step)

h = input()
av = input()
bv = input()
ah = pointwise(av, h; f='mul')
h1 = pointwise(ah, bv; f='add')


The step is not a callback — it is a value: printable, runnable on its own,
countable, differentiable. That is what makes the adjoint derivable later.

Folding it should agree with notebook 07's `linrec` composite reducer
exactly, forward *and* backward. Same recurrence, two combinators, two
entirely different code paths through the AD transform:

In [4]:
linrec = defreducer(
    "linrec",
    state=2,
    element=2,
    lift=lambda a, b: (a, b),
    combine=lambda left, right: (left[0] * right[0], right[0] * left[1] + right[1]),
    init=(1.0, 0.0),
    project=lambda A, B: B,
)
n = 5
sc_inputs = {
    "h0": T(0.0, ()),
    "a": T(rng.uniform(0.5, 1.1, n), ("t",)),
    "b": T(rng.standard_normal(n), ("t",)),
}


def scalar_prog(kind):
    if kind == "fold":
        head = (
            instr("h0", "input"),
            instr("a", "input"),
            instr("b", "input"),
            instr(
                "h",
                "fold",
                ["h0", "a", "b"],
                step=step,
                dim="t",
                state=("h",),
                element=("av", "bv"),
                carry={"h": "h1"},
                out=("emit", "h1"),
            ),
        )
    else:
        head = (
            instr("a", "input"),
            instr("b", "input"),
            instr("h", "scan", ["a", "b"], f="linrec", dim="t"),
        )
    tail = (
        instr("hh", "pointwise", ["h", "h"], f="mul"),
        instr("loss", "reduce", ["hh"], f="sum", dims=("t",)),
    )
    return Program(head + tail)


got = {}
for kind in ("fold", "scan"):
    jp, gr = grad(scalar_prog(kind), "loss", sc_inputs)
    env = run(jp, sc_inputs)
    got[kind] = (env["h"].to_numpy(), env[gr["a"]].to_numpy(), env[gr["b"]].to_numpy())
for i, what in enumerate(("h", "dL/da", "dL/db")):
    print(f"  {what:<6} max|fold − scan| = {np.abs(got['fold'][i] - got['scan'][i]).max():.2e}")

  h      max|fold − scan| = 0.00e+00
  dL/da  max|fold − scan| = 0.00e+00
  dL/db  max|fold − scan| = 0.00e+00


## A matrix state: gated linear attention

Now the state that motivates the combinator. Gated linear attention
(Mamba-2 / DeltaNet-style) carries a **matrix** S over key dim `p` and value
dim `r`:

    S_t = a_t · S_{t−1} + k_t v_tᵀ        y_t = S_tᵀ q_t

The outer product and the readout are ordinary tensor instructions —
repeats and a reduce, exactly as in notebook 05's matmul — so the step is a
15-instruction program with no new machinery at all.

In [5]:
DK, DV, TN = 3, 2, 4

GLA_STEP = Program(
    (
        instr("S", "input"),  # state: (p, r)
        instr("a", "input"),  # elements: this step's slices
        instr("kk", "input"),
        instr("vv", "input"),
        instr("qq", "input"),
        instr("a1", "repeat", ["a"], name="p", extent=(0, DK)),
        instr("ar", "repeat", ["a1"], name="r", extent=(0, DV)),
        instr("kr", "repeat", ["kk"], name="r", extent=(0, DV)),
        instr("vr", "repeat", ["vv"], name="p", extent=(0, DK)),
        instr("Sa", "pointwise", ["ar", "S"], f="mul"),  # a_t · S
        instr("kv", "pointwise", ["kr", "vr"], f="mul"),  # outer product k v^T
        instr("S1", "pointwise", ["Sa", "kv"], f="add"),  # the new state
        instr("qr", "repeat", ["qq"], name="r", extent=(0, DV)),
        instr("Sq", "pointwise", ["S1", "qr"], f="mul"),  # readout S^T q
        instr("y", "reduce", ["Sq"], f="sum", dims=("p",)),
    )
)

gla = Program(
    (
        instr("S0", "input"),
        instr("a", "input"),
        instr("k", "input"),
        instr("v", "input"),
        instr("q", "input"),
        instr(
            "ys",
            "fold",
            ["S0", "a", "k", "v", "q"],
            step=GLA_STEP,
            dim="t",
            state=("S",),
            element=("a", "kk", "vv", "qq"),
            carry={"S": "S1"},
            out=("emit", "y"),
        ),
        instr("y2", "pointwise", ["ys", "ys"], f="mul"),
        instr("loss", "reduce", ["y2"], f="sum", dims=("t", "r")),
    )
)

gla_inputs = {
    "S0": T(rng.standard_normal((DK, DV)), ("p", "r")),
    "a": T(rng.uniform(0.5, 1.0, TN), ("t",)),
    "k": T(rng.standard_normal((TN, DK)), ("t", "p")),
    "v": T(rng.standard_normal((TN, DV)), ("t", "r")),
    "q": T(rng.standard_normal((TN, DK)), ("t", "p")),
}

The step runs on its own — feed it one timestep's slices and it is just a
program:

In [6]:
one = run(
    GLA_STEP,
    {
        "S": gla_inputs["S0"],
        "a": gla_inputs["a"].select(t=0),
        "kk": gla_inputs["k"].select(t=0),
        "vv": gla_inputs["v"].select(t=0),
        "qq": gla_inputs["q"].select(t=0),
    },
)
S0n, kn, vn, qn = (gla_inputs[nm].to_numpy() for nm in ("S0", "k", "v", "q"))
S1n = gla_inputs["a"].to_numpy()[0] * S0n + np.outer(kn[0], vn[0])
print("one step, standalone:  max|Δ| =", np.abs(one["S1"].to_numpy(order=("p", "r")) - S1n).max())
print("its readout y        :", np.round(one["y"].to_numpy(), 6), " vs ", np.round(S1n.T @ qn[0], 6))

one step, standalone:  max|Δ| = 0.0
its readout y        : [ 0.480774 -1.192476]  vs  [ 0.480774 -1.192476]


…and folding it over the sequence reproduces the recurrence:

In [7]:
S = S0n.copy()
av = gla_inputs["a"].to_numpy()
ref = []
for t in range(TN):
    S = av[t] * S + np.outer(kn[t], vn[t])
    ref.append(S.T @ qn[t])
ref = np.stack(ref)

env = run(gla, gla_inputs)
ys = env["ys"].to_numpy(order=("t", "r"))
print("fold vs the recurrence:  max|Δ| =", f"{np.abs(ys - ref).max():.2e}")
print("emitted layout        :", env["ys"].names, ys.shape, " (the scan dim, then the step output's dims)")

fold vs the recurrence:  max|Δ| = 4.44e-16
emitted layout        : ('t', 'r') (4, 2)  (the scan dim, then the step output's dims)


## The contracts

Three refusals hold the combinator together. The **carry** must return the
state at its exact layout — a step that quietly shrinks the state would
make the recurrence ill-typed after one iteration, so it is rejected on the
spot rather than reshaped. `out=("final", v)` must name a carry var (only
carried state survives the loop). And the scan dim must be **chartless**:
physical time labels get stripped going in and glued back on the result,
so the lattice algebra inside never sees a chart.

In [8]:
shrinking = Program((instr("E", "input"), instr("E1", "slice", ["E"], ranges={"x": (0, 5)})))
drift = Program(
    (
        instr("E0", "input"),
        instr(
            "Ef",
            "fold",
            ["E0"],
            step=shrinking,
            dim="t",
            state=("E",),
            element=(),
            carry={"E": "E1"},
            out=("final", "E1"),
            extent=(0, 2),
        ),
    )
)
try:
    run(drift, {"E0": T(np.zeros(6), ("x",))})
except ValueError as e:
    print("refused:", e)

not_a_carry = {**dict(gla.instrs[5].params), "out": ("final", "y")}
try:
    run(Program(gla.instrs[:5] + (instr("ys", "fold", gla.instrs[5].operands, **not_a_carry),)), gla_inputs)
except ValueError as e:
    print("refused:", e)

try:
    run(gla, {**gla_inputs, "a": gla_inputs["a"].with_charts(t=("0 s", "1 s"))})
except ValueError as e:
    print("refused:", e)

refused: fold carry 'E' changes the state layout: [('x', 0, 6)] -> [('x', 0, 5)]
refused: fold out=('final', v) requires v to be a carry output
refused: fold scan dim 't' must be chartless/unlabeled (strip_charts first)


## Backward: the adjoint derives itself

Here is the move. The step is a program, and `grad` transforms programs —
so the fold's backward pass is obtained by **self-application**:

1. wrap the step with one cotangent input per state and per output, and a
   scalarized target (the sum of ⟨cot, out⟩ inner products);
2. run `grad` on that wrapper — the joint program *is* the backward step,
   mapping (s_{t−1}, e_t, ȳ_t, ŝ_t) to (ŝ_{t−1}, ē_t);
3. fold **that** in reversed time, carrying ŝ.

Nothing about the step is inspected. Standard BPTT with per-step recompute,
generated rather than hand-written — and every gradient below is checked
against finite differences.

In [9]:
for wrt in ("S0", "a", "k", "v", "q"):
    validate(gla, gla_inputs, wrt)

OK   dloss/dS0  max|Δ| = 1.31e-08
OK   dloss/da   max|Δ| = 4.05e-09
OK   dloss/dk   max|Δ| = 1.43e-08
OK   dloss/dv   max|Δ| = 1.64e-08


OK   dloss/dq   max|Δ| = 9.16e-09

The generated program shows the shape of it: one fold re-emits the forward
state trajectory (needed as s_{t−1} for each backward step), then one
reverse fold per requested cotangent — `emit` for element gradients,
`final` for the init's. The state names of those backward folds are the
cotangent inputs the wrapper introduced (`%ct0`), and their elements are
the forward state, the forward elements, and the output cotangent.

In [10]:
joint, grads = grad(gla, "loss", dict(gla_inputs))
folds = [i for i in joint.instrs if i.op == "fold"]
print(f"{len(folds)} folds in the joint program:")
for i in folds:
    p = i.params
    print(f"  {i.var:<6} state={p['state']}  element={p['element']}  out={p['out']}")
print()
back = folds[-1].params["step"]
print("the derived backward step program:", len(back.instrs), "instructions, e.g.")
for x in list(back.instrs)[-5:]:
    print("   ", x)

7 folds in the joint program:
  ys     state=('S',)  element=('a', 'kk', 'vv', 'qq')  out=('emit', 'y')
  %g15   state=('S',)  element=('a', 'kk', 'vv', 'qq')  out=('emit', 'S1')
  %g31   state=('%ct0',)  element=('S', 'a', 'kk', 'vv', 'qq', '%ct3')  out=('emit', '%g39')
  %g34   state=('%ct0',)  element=('S', 'a', 'kk', 'vv', 'qq', '%ct3')  out=('emit', '%st36')
  %g37   state=('%ct0',)  element=('S', 'a', 'kk', 'vv', 'qq', '%ct3')  out=('emit', '%st34')
  %g40   state=('%ct0',)  element=('S', 'a', 'kk', 'vv', 'qq', '%ct3')  out=('emit', '%st21')
  %g43   state=('%ct0',)  element=('S', 'a', 'kk', 'vv', 'qq', '%ct3')  out=('final', '%st32')

the derived backward step program: 62 instructions, e.g.
    %g35 = reduce(%st26; f='sum', dims=('r',))
    %st36 = with_charts(%g35; charts={'p': None})
    %g37 = reduce(%st30; f='sum', dims=('r',))
    %st38 = with_charts(%g37; charts={'p': None})
    %g39 = reduce(%st38; f='sum', dims=('p',))


## A PDE: FDTD leapfrog

The other half of the design space: **two** states, **no** elements. A 1D
finite-difference time-domain solver leapfrogs the electric and magnetic
fields on staggered grids,

    H ← H + c·(E[x+1] − E[x])        E ← E + c·(H[x] − H[x−1])

which is shift / slice / pad / pointwise — layout ops and arithmetic, the
vocabulary of notebooks 02–04. With no element tensors the fold is
`extent`-driven: time is a loop count, not a data axis.

In [11]:
NX, NT, C = 24, 20, 0.5


def fdtd_step(nx, c):
    return Program(
        (
            instr("E", "input"),
            instr("H", "input"),
            instr("Es", "shift", ["E"], deltas={"x": -1}),
            instr("Ea", "slice", ["Es"], ranges={"x": (0, nx - 1)}),
            instr("Eb", "slice", ["E"], ranges={"x": (0, nx - 1)}),
            instr("dE", "pointwise", ["Ea", "Eb"], f="sub"),
            instr("cH", "const", [], value=c, dims=(("x", (0, nx - 1)),)),
            instr("dHs", "pointwise", ["cH", "dE"], f="mul"),
            instr("H1", "pointwise", ["H", "dHs"], f="add"),  # the new H
            instr("Hs", "shift", ["H1"], deltas={"x": 1}),
            instr("Ha", "slice", ["H1"], ranges={"x": (1, nx - 1)}),
            instr("Hb", "slice", ["Hs"], ranges={"x": (1, nx - 1)}),
            instr("dH", "pointwise", ["Ha", "Hb"], f="sub"),
            instr("pdH", "pad", ["dH"], fill=0.0, extents={"x": (0, nx)}),
            instr("cE", "const", [], value=c, dims=(("x", (0, nx)),)),
            instr("dEs", "pointwise", ["cE", "pdH"], f="mul"),
            instr("E1", "pointwise", ["E", "dEs"], f="add"),  # the new E
        )
    )


def fdtd(out=("emit", "E1"), steps=NT, nx=NX, c=C):
    dims = ("x",) if out[0] == "final" else ("t", "x")
    return Program(
        (
            instr("E0", "input"),
            instr("H0", "input"),
            instr(
                "Ef",
                "fold",
                ["E0", "H0"],
                step=fdtd_step(nx, c),
                dim="t",
                state=("E", "H"),
                element=(),
                carry={"E": "E1", "H": "H1"},
                out=out,
                extent=(0, steps),
            ),
            instr("E2", "pointwise", ["Ef", "Ef"], f="mul"),
            instr("loss", "reduce", ["E2"], f="sum", dims=dims),
        )
    )


E0 = np.exp(-0.5 * ((np.arange(NX) - 11) / 1.5) ** 2)  # a Gaussian pulse, mid-domain
fields = {"E0": T(E0, ("x",)), "H0": T(np.zeros(NX - 1), ("x",))}

E, H, loop = E0.copy(), np.zeros(NX - 1), []
for _ in range(NT):
    H = H + C * (E[1:] - E[:-1])
    dH = np.zeros(NX)
    dH[1 : NX - 1] = H[1 : NX - 1] - H[0 : NX - 2]
    E = E + C * dH
    loop.append(E.copy())

traj = run(fdtd(), fields)["Ef"].to_numpy(order=("t", "x"))
print("fold vs the time loop:  max|Δ| =", f"{np.abs(traj - np.stack(loop)).max():.2e}")

fold vs the time loop:  max|Δ| = 0.00e+00


`out=("emit", "E1")` stacks every step's field into a (t, x) tensor — the
whole space-time trajectory as one ordinary tensor. The pulse splits into two
counter-propagating halves that walk out to the boundaries:

In [12]:
ramp = " .:-=+*#%@"
scale = np.abs(traj).max()
print("|E|(x, t):")
for t, row in enumerate(traj):
    line = "".join(ramp[int(abs(x) / scale * (len(ramp) - 1))] for x in row)
    print(f"  t={t:2d} |{line}|")

final = run(fdtd(out=("final", "E1")), fields)["Ef"]
print()
print("out=('final', 'E1') keeps only the last field:", final.names, " max|Δ| vs traj[-1] =", end=" ")
print(f"{np.abs(final.to_numpy() - traj[-1]).max():.2e}")

|E|(x, t):
  t= 0 |        .=#@#=.         |
  t= 1 |        :=*#*=:         |
  t= 2 |       .-+++++-.        |
  t= 3 |       :=+=-=+=:        |
  t= 4 |      .-==:.:==-.       |
  t= 5 |      :=+-. .-+=:       |
  t= 6 |     .-==:   :==-.      |
  t= 7 |     :=+-.   .-+=:      |
  t= 8 |    .-==:     :==-.     |
  t= 9 |    :=+-.     .-+=:     |
  t=10 |   .-==:       :==-.    |
  t=11 |   :=+-.       .-+=:    |
  t=12 |  .-==:         :==-.   |
  t=13 |  :===.         .===:   |
  t=14 | .-==-           -==-.  |
  t=15 | .-==.           .==-:  |
  t=16 | :==-             -==-. |
  t=17 | :==.             .==-. |
  t=18 | -=-               -==: |
  t=19 | :-.               :==: |

out=('final', 'E1') keeps only the last field: ('x',)  max|Δ| vs traj[-1] = 0.00e+00


And the gradient of a wave simulation with respect to its initial
conditions — the adjoint-state method — is what the derived reverse fold
computes, for both output forms:

In [13]:
for out in (("final", "E1"), ("emit", "E1")):
    prog = fdtd(out=out, steps=6)
    print(f"out={out}")
    validate(prog, fields, "E0")
    validate(prog, fields, "H0")

out=('final', 'E1')


OK   dloss/dE0  max|Δ| = 1.70e-10


OK   dloss/dH0  max|Δ| = 1.47e-10
out=('emit', 'E1')


OK   dloss/dE0  max|Δ| = 1.28e-09


OK   dloss/dH0  max|Δ| = 1.31e-09


## The empty fold

Zero steps is the identity on the state, and its adjoint passes the
cotangent straight through — no special-casing at the call site, which is
what makes `fold` safe to generate mechanically.

In [14]:
p0 = fdtd(out=("final", "E1"), steps=0)
print("zero steps returns the init:  max|Δ| =", f"{np.abs(run(p0, fields)['Ef'].to_numpy() - E0).max():.2e}")
jp, gr = grad(p0, "loss", fields)
ej = run(jp, fields)
print("dL/dE0 = 2·E0             :  max|Δ| =", f"{np.abs(ej[gr['E0']].to_numpy() - 2 * E0).max():.2e}")
print("dL/dH0 = 0                :  max|Δ| =", f"{np.abs(ej[gr['H0']].to_numpy()).max():.2e}")

zero steps returns the init:  max|Δ| = 0.00e+00
dL/dE0 = 2·E0             :  max|Δ| = 0.00e+00
dL/dH0 = 0                :  max|Δ| = 0.00e+00


## Costs scale with steps

`ops_count` prices a fold as the step's tally times the number of steps —
the step program is data, so nothing is opaque here either.

In [15]:
ops = ops_count(gla, gla_inputs)
per_mul, per_add = 3 * DK * DV, DK * DV + (DK - 1) * DV
print("fold instruction  :", dict(ops.per_var["ys"]))
print(f"expected per step : mul {per_mul}, add {per_add}   × {TN} steps")
print("whole program     :", dict(ops.total))

fold instruction  : {'mul': 72, 'add': 40}
expected per step : mul 18, add 10   × 4 steps
whole program     : {'mul': 80, 'add': 47}


## Units survive the carry

The signature pass treats the carry as a fixed point: seed the state's
declared units, infer the step, merge into the state, repeat until stable.
A recurrence that would add volts to amperes is refused before it runs.

In [16]:
V = u.define("V", dim="voltage")
A = u.define("A", dim="current")
united = {
    "h0": T(0.0, ()).with_value_units(V),
    "a": T(np.ones(n), ("t",)),  # dimensionless gate
    "b": T(np.ones(n), ("t",)).with_value_units(V),
}
sigs = infer_signatures(scalar_prog("fold"), united)
print("h    :", sigs["h"])
print("loss :", sigs["loss"], " (a sum of squares, hence V²)")

try:
    infer_signatures(scalar_prog("fold"), {**united, "b": T(np.ones(n), ("t",)).with_value_units(A)})
except SignatureError as e:
    print("refused:", e)

h    : VInfo(carrier='real', unit=V)
loss : VInfo(carrier='real', unit=V*V)  (a sum of squares, hence V²)
refused: fold.carry: unit mismatch — V vs A


---
Notes and known gaps (CONCERNS #24). The reference `fold` is sequential and
its adjoint stores everything: forward state trajectories are re-emitted
per state component and held whole (store-everything BPTT — the
checkpointing work in LEVELS.md L1 is where that trade-off gets made
properly), and each element/init cotangent re-runs the reverse fold, one
fold per output. `out=("final", v)` is restricted to carry vars. The scan
dim must be chartless in the reference layer. An associative tensor
*combine* — the chunk-parallel license Mamba-2 exploits — is a declaration
a compiler may add later without changing this denotation. Second order
through `fold` is untested.